### Imports and Configuration

In [13]:
import os
import numpy as np
import pandas as pd
import warnings
from scipy.stats import linregress
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')


### Paths

In [14]:
out_dir = "../data/procdata/3_carboplatin"

## Input files
response_file = "../data/procdata/3_carboplatin/McGill_batch_response_table.csv"
omicsmap = "../data/procdata/3_carboplatin/McGill_modToBiobaseMap.csv"
mutation_file = "../data/procdata/3_carboplatin/McGill_mutation_matrix.csv"
cnv_file = "../data/procdata/3_carboplatin/McGill_cnv_matrix.csv"

## Output files
models_out = "../data/procdata/3_carboplatin/McGill_cisplatin_models.tsv"
mut_out = "../data/procdata/3_carboplatin/McGill_mutation_cisplatin_filtered.csv"
cnv_out = "../data/procdata/3_carboplatin/McGill_cnv_cisplatin_filtered.csv"

## NestVNN files
nest_dir = "../data/procdata/3_carboplatin/McGill_nest_input"
gene_file = "../data/rawdata/nest_vnn/gene2ind.txt"
mut_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/cell2mutation.txt"
cna_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/cell2amplification.txt"
cnd_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/cell2cndeletion.txt"
cell_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/cell2ind.txt"
gene_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/gene2ind.txt"
test_data_vnn = "../data/procdata/3_carboplatin/McGill_nest_input/test_data.txt"

### Load response tabel

In [15]:
resp = pd.read_csv(response_file)

print("Total experiments:", resp.shape[0])

Total experiments: 128


### Parse model and drug


In [16]:
resp[["model_id", "drug"]] = resp["batch.name"].str.split(".", expand=True)

print("Unique drugs:", resp["drug"].unique())

Unique drugs: <StringArray>
['Cisplatin', 'Gemcitabine', 'Paclitaxel', 'Doxorubicin', 'Carboplatin']
Length: 5, dtype: str


### Filter cisplatin/carboplatin experiments

In [17]:
cis = resp[resp["drug"].str.lower() == "cisplatin"].copy()
car = resp[resp["drug"].str.lower() == "carboplatin"].copy()
print("Cisplatin experiments:", cis.shape[0])
print("Carboplatin experiments:", car.shape[0])

Cisplatin experiments: 30
Carboplatin experiments: 12


Carboplatin data is two low (n=12) for robust ML model testing. 

Will focuse on cisplatin for model testing.

### Extract unique models

In [18]:
models = sorted(cis["model_id"].unique())

print("Unique cisplatin models:", len(models))

models_df = pd.DataFrame({"model_id": models})

Unique cisplatin models: 30


### Extract omics map

In [19]:
omics = pd.read_csv(omicsmap)
omics = omics[omics["source"] == "PDX"]
omics = omics[omics["model.id"].str.lower().str.endswith("ci")]
omics["model_id"] = "GCRC" + omics["model.id"].str.split(".").str[2].str.replace("Ci", "", regex=False)
# collapse mDataType
omics = (
    omics
    .groupby(["model.id", "biobase.id", "source", "model_id"], as_index=False)
    .agg({"mDataType": lambda x: "/".join(sorted(x.unique()))})
)
omics = omics[omics["mDataType"].str.contains("CNV")].reset_index(inplace=False)

omics = omics[["model_id", "biobase.id"]]
omics = omics.rename(columns={"biobase.id": "omics_id"})

print(omics)

    model_id      omics_id
0   GCRC1834   SA1028X3_3L
1   GCRC2029   SA1037X1_2L
2   GCRC1828   SA1027X1_1L
3   GCRC1945   SA1033X1_2R
4   GCRC1735   SA920X2_23R
5   GCRC1738  SA1030X2_17L
6   GCRC2076    SA1041X2_1
7   GCRC1851    SA994X2_3L
8   GCRC2006   SA1035X1_8L
9   GCRC1971   SA1019X1_1L
10  GCRC1979   SA1032X1_2R
11  GCRC1876   SA1017X1_4L
12  GCRC1915   SA1023X1_2L
13  GCRC1924    SA993X2_5R
14  GCRC1939   SA1024X1_1L
15  GCRC1868   SA1016X3_3L
16  GCRC2001   SA1026X1_4R
17  GCRC1863   SA1025X1_2R
18  GCRC1882   SA1021X2_1L
19  GCRC1944   SA1031X1_1R
20  GCRC1963    SA992X1_4R
21  GCRC2061   SA1040X1_1L
22  GCRC2047   SA1038X1_2L
23  GCRC1887    SA995X1_5R
24  GCRC1715  SA1029X2_14L


### Save model list

In [20]:
cis = cis.merge(omics, on="model_id", how="inner")
cis.to_csv(models_out, sep="\t", index=False)

print("Saved model list to:", models_out)

Saved model list to: ../data/procdata/3_carboplatin/McGill_cisplatin_models.tsv


### Filter Omics Data for Cisplatin Models

In [21]:
### Filter mutation and CNV matrices to only include cisplatin models
mut = pd.read_csv(mutation_file, index_col=0)
cnv = pd.read_csv(cnv_file, index_col=0)
models_ext = cis
print("Mutation matrix shape:", mut.shape)
print("CNV matrix shape:", cnv.shape)

# Build mapping: omics_id → model_id
sample_map = dict(zip(models_ext["omics_id"], models_ext["model_id"]))
print("Mapped samples:", len(sample_map))

# Rename omics columns to model_id
mut = mut.rename(columns=sample_map)
cnv = cnv.rename(columns=sample_map)

# Keep only columns corresponding to model_id
model_ids = sorted(models_ext["model_id"].unique())
mut = mut[mut.columns.intersection(model_ids)]
cnv = cnv[cnv.columns.intersection(model_ids)]
print("Mutation columns after renaming:", mut.shape[1])
print("CNV columns after renaming:", cnv.shape[1])

# Ensure both matrices share same models
common_models = sorted(set(mut.columns) & set(cnv.columns))
print("Models with mutation + CNV:", len(common_models))

mut_filtered = mut[common_models]
cnv_filtered = cnv[common_models]
print("Filtered mutation matrix:", mut_filtered.shape)
print("Filtered CNV matrix:", cnv_filtered.shape)

# Save filtered matrices
mut_filtered.to_csv(mut_out)
cnv_filtered.to_csv(cnv_out)

print("Saved:")
print(mut_out)
print(cnv_out)

Mutation matrix shape: (4876, 70)
CNV matrix shape: (27090, 70)
Mapped samples: 25
Mutation columns after renaming: 25
CNV columns after renaming: 25
Models with mutation + CNV: 25
Filtered mutation matrix: (4876, 25)
Filtered CNV matrix: (27090, 25)
Saved:
../data/procdata/3_carboplatin/McGill_mutation_cisplatin_filtered.csv
../data/procdata/3_carboplatin/McGill_cnv_cisplatin_filtered.csv


### Prepare for Nest-VNN

In [22]:
# Define the explicit functional terms (the whitelist)
functional_whitelist = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Site",
    "Translation_Start_Site",
    "Nonstop_Mutation"
}

def binarize_mutation_inclusive(val):
    """
    Binarizes mutation data by checking for the presence of 
    explicitly functional variant classifications.
    """
    if pd.isna(val) or val == "" or str(val).lower() == "nan":
        return 0
    
    # Split by comma for cases where multiple mutations are collapsed into one cell
    # Strip whitespace and ensure we handle potential case sensitivity issues
    parts = {p.strip() for p in str(val).split(',')}
    
    # INCLUSIVE LOGIC: If ANY of the parts are in the whitelist, it's a 1
    if any(p in functional_whitelist for p in parts):
        return 1
    
    # Otherwise (e.g., only "Silent", "Intron", or unknown terms), return 0
    return 0

In [ ]:
# Load in nesv_vnn genes
gene_map = pd.read_csv(gene_file, sep="\t", header=None)
gene_map.to_csv(gene_vnn, sep="\t", header=False, index=False)
gene_map.columns = ["index", "gene"]
gene_list = gene_map["gene"].tolist()
print("NEST_VNN Gene:", len(gene_list))

# Filter to NeST genes
mut = mut_filtered[mut_filtered.index.isin(gene_list)]
cnv = cnv_filtered[cnv_filtered.index.isin(gene_list)]

print("\nAfter NeST gene filter")
print("Mutation:", mut.shape)
print("CNV:", cnv.shape)

# Prepare mutation matrix
#mut_bin = mut.map(lambda x: 0 if (pd.isna(x) or x == "0" or str(x).lower() == "silent") else 1)
mut_bin = mut.map(binarize_mutation_inclusive)
mut_bin = mut_bin.reindex(gene_list).fillna(0)

# Prepare CNV matrices
cna = (cnv == 2).astype(int)
cnd = (cnv == -2).astype(int)
cna = cna.reindex(gene_list).fillna(0)
cnd = cnd.reindex(gene_list).fillna(0)

# transpose
mut_bin = mut_bin.T
cna = cna.T
cnd = cnd.T

# convert to int
mut_bin = mut_bin.astype(int)
cna = cna.astype(int)
cnd = cnd.astype(int)

# Save NeST input files
mut_bin.to_csv(mut_vnn, header=False, index=False)
cna.to_csv(cna_vnn, header=False, index=False)
cnd.to_csv(cnd_vnn, header=False, index=False)

# Create cell2ind.txt
cells = mut_bin.index.tolist()

cell2ind = pd.DataFrame({
    "index": range(len(cells)),
    "cell": cells
})

cell2ind.to_csv(cell_vnn, sep="\t", header=False, index=False)

# Create test_data.txt
test = pd.DataFrame({
    "cell": cells,
    "drug": "Cisplatin",
    "test" : 0,
    "model": "PDX"
})

test.to_csv(test_data_vnn, sep="\t", header=False, index=False)

# Sanity check

print("Mutation matrix:", mut_bin.shape)
print("CNA matrix:", cna.shape)
print("CND matrix:", cnd.shape)


NEST_VNN Gene: 718

After NeST gene filter
Mutation: (235, 25)
CNV: (717, 25)
Mutation matrix: (25, 718)
CNA matrix: (25, 718)
CND matrix: (25, 718)


### Statistics

In [12]:
def matrix_stats(name, df):

    total = df.size
    nonzero = (df != 0).sum().sum()
    sparsity = 1 - nonzero / total

    print(f"\n{name} statistics")
    print("----------------------------")
    print("Models:", df.shape[0])
    print("Genes:", df.shape[1])
    print("Total entries:", total)
    print("Nonzero entries:", nonzero)
    print("Sparsity:", round(sparsity, 4))

    model_events = (df != 0).sum(axis=1)
    gene_events = (df != 0).sum(axis=0)

    print("\nPer model events")
    print("Median:", model_events.median())
    print("Mean:", round(model_events.mean(),2))
    print("Max:", model_events.max())

    print("\nPer gene events")
    print("Median:", gene_events.median())
    print("Mean:", round(gene_events.mean(),2))
    print("Max:", gene_events.max())

matrix_stats("Mutation", mut_bin)
matrix_stats("CNA", cna)
matrix_stats("CND", cnd)


Mutation statistics
----------------------------
Models: 25
Genes: 718
Total entries: 17950
Nonzero entries: 215
Sparsity: 0.988

Per model events
Median: 7.0
Mean: 8.6
Max: 38

Per gene events
Median: 0.0
Mean: 0.3
Max: 19

CNA statistics
----------------------------
Models: 25
Genes: 718
Total entries: 17950
Nonzero entries: 1901
Sparsity: 0.8941

Per model events
Median: 77.0
Mean: 76.04
Max: 148

Per gene events
Median: 2.0
Mean: 2.65
Max: 18

CND statistics
----------------------------
Models: 25
Genes: 718
Total entries: 17950
Nonzero entries: 1401
Sparsity: 0.9219

Per model events
Median: 27.0
Mean: 56.04
Max: 213

Per gene events
Median: 1.0
Mean: 1.95
Max: 10
